<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/SentenceEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel
import random

In [ ]:
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

In [ ]:
sample = ["The cat sat on the mat.", "Attention is all you need."]

enc = tokenizer(sample, padding=True, truncation=True, return_tensors="pt")

print("input_ids shape:", enc["input_ids"].shape)
print("attention_mask shape:", enc["attention_mask"].shape)

print("\nTokens for sentence 0:")
print(tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))

In [ ]:
with torch.no_grad():
    enc = enc.to(device)
    out = model(**enc)

# last_hidden_state: (batch, seq_len, hidden_size)
print("Token embeddings shape:", out.last_hidden_state.shape)

In [ ]:
def mean_pool(token_embeddings, attention_mask):
    mask =

    summed =

    counts =

    return summed / counts

with torch.no_grad():
    sentence_emb = mean_pool(out.last_hidden_state, enc["attention_mask"])
print("Sentence embeddings shape:", sentence_emb.shape)

In [ ]:
@torch.no_grad()
def encode(texts, batch_size=64):
    # encode a list of strings into L2-normalized sentence embeddings
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # tokenize batch
        enc =

        out =

        # mean pool
        emb =
        # L2 normalize
        emb =

        all_emb.append(emb.cpu())

    return torch.cat(all_emb, dim=0)

# similar sentences should have high cosine sim
test = [
    "I love pizza.",
    "Pizza is my favorite food.",
    "The stock market crashed today.",
]

e = encode(test)

print("Norms (should all be 1.0):", e.norm(dim=1))

# cosine similarity
sim =

print("\nSimilarity matrix:")
print(sim.numpy().round(3))

In [ ]:
from datasets import load_dataset

ds = load_dataset("ag_news", split="train")
N_CORPUS = 5000
indices = random.sample(range(len(ds)), N_CORPUS)
corpus_texts = [ds[i]["text"] for i in indices]
corpus_labels = [ds[i]["label"] for i in indices]
label_names = ["World", "Sports", "Business", "Sci/Tech"]

In [ ]:
corpus_emb = encode(corpus_texts, batch_size=128)
print(f"Corpus embedding shape: {corpus_emb.shape}")

In [ ]:
def search(query, corpus_emb, corpus_texts, k=5):
    q =

    scores =

    top =

    return ...

for query in [
    "soccer world cup final",
    "new smartphone release",
    "stock market crash",
    "elections in europe",
]:
    print(f"\nQuery: {query}")
    for score, text in search(query, corpus_emb, corpus_texts, k=3):
        print(f"  {score:.3f}  {text}")

In [ ]:
def search_with_examples(example_texts, corpus_emb, corpus_texts, k=5):
    example_emb =

    # average embeddings
    prototype =
    # renormalize
    prototype =

    scores =

    top =

    return ...

# define a concept by multiple examples, not a single query
space_examples = [
    "NASA launched a new satellite into orbit.",
    "Astronauts returned from the space station.",
    "The Mars rover sent back new photos.",
]

for score, text in search_with_examples(space_examples, corpus_emb, corpus_texts, k=5):
    print(f"{score:.3f}  {text}")

In [ ]:
def odd_one_out(texts):
    emb =
    sim =

    # for each item, average similarity to *others* (exclude self)
    sim_no_self =

    mean_sim =
    odd_idx =

    return odd_idx, mean_sim

candidates = [
    "The cat is sleeping on the couch.",
    "My dog loves to play fetch in the park.",
    "Our hamster runs on its wheel every night.",
    "The quarterly earnings report exceeded expectations.",
    "I just adopted a kitten from the shelter.",
]

idx, scores = odd_one_out(candidates)
for i, (t, s) in enumerate(zip(candidates, scores.tolist())):
    mark = " | odd one out" if i == idx else ""
    print(f"{s:.3f}  {t}{mark}")

In [ ]:
import umap
import matplotlib.pyplot as plt

reducer =
coords =

plt.figure(figsize=(9, 7))
for label_id, name in enumerate(label_names):
    mask = np.array(corpus_labels) == label_id
    plt.scatter(coords[mask, 0], coords[mask, 1], s=4, alpha=0.5, label=name)
plt.legend(markerscale=3)
plt.title("AG News corpus embeddings (MiniLM, UMAP to 2D)")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.show()

In [ ]:
K = 4
N_ITERS = 20

# pick K random points from the corpus as starting prototypes
init_indices =
centroids =

for it in range(N_ITERS):
    # assignment step: each point -> nearest centroid (by cosine sim)
    scores =
    assignments =

    # update step: new centroid = mean of assigned points, then renormalize
    new_centroids = torch.zeros_like(centroids)
    for k in range(K):
        mask =

        # make sure k has assignments
        if mask.sum() > 0:
            new_centroids[k] =
        else:
            # empty cluster, reinitialize randomly
            new_centroids[k] =

    # normalizw
    new_centroids =

    # check convergence, how much did centroids move?
    shift =

    centroids = new_centroids

    print(f"iter {it+1:2d} max centroid shift: {shift:.4f}")
    if shift < 1e-4:
        print("converged")
        break

# cluster sizes
print("\nCluster sizes:")
for k in range(K):
    print(f"  cluster {k}: {(assignments == k).sum().item()} docs")

In [ ]:
print("Label composition of each cluster:")
print(f"{'cluster':>8} | " + " ".join(f"{n:>10s}" for n in label_names))
true_labels = np.array(corpus_labels)
for k in range(K):
    mask = (assignments == k).numpy()

    counts = [((true_labels == c) & mask).sum() for c in range(len(label_names))]

    total = sum(counts)

    pct = [100*c/total if total > 0 else 0 for c in counts]

    print(f"{k:>8} | " + " ".join(f"{p:>9.1f}%" for p in pct))

cluster_to_label = []
for k in range(K):
    mask = (assignments == k).numpy()

    if mask.sum() > 0:
        dominant = np.bincount(true_labels[mask], minlength=len(label_names)).argmax()
    else:
        dominant = 0

    cluster_to_label.append(dominant)

mapped_preds = np.array([cluster_to_label[a] for a in assignments.tolist()])
purity = (mapped_preds == true_labels).mean()

print(f"\nCluster purity: {100*purity:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# left: true labels
for label_id, name in enumerate(label_names):
    mask = np.array(corpus_labels) == label_id
    axes[0].scatter(coords[mask, 0], coords[mask, 1], s=4, alpha=0.5, label=name)
axes[0].legend(markerscale=3)
axes[0].set_title("True labels (held out)")

# right: our cluster assignments
for k in range(K):
    mask = (assignments == k).numpy()
    axes[1].scatter(coords[mask, 0], coords[mask, 1], s=4, alpha=0.5, label=f"cluster {k}")
axes[1].legend(markerscale=3)
axes[1].set_title(f"K-means clusters (K={K}, no labels used)")

plt.show()